In [38]:
import pandas as pd
import numpy as np
import boto3
from sklearn.model_selection import train_test_split
import sagemaker
from sagemaker import Session
import io
import sagemaker.amazon.common as smac
import os
from sagemaker.amazon.amazon_estimator import get_image_uri
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

In [7]:
df = pd.read_csv("student_scores.csv")
df.head()

,Hours,Scores
0,2.5,21
1,5.1,47
2,3.2,27
3,8.5,75
4,3.5,30


In [8]:
df.shape

(25, 2)

In [9]:
x=df[["Hours"]]
y=df[["Scores"]]

In [10]:
x.dtypes
x=x.astype("float32")
y=y.astype("float32")

In [11]:
y.dtypes

Scores    float32
dtype: object

In [12]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2)

In [13]:
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

In [14]:
X_train

,Hours
0,7.4
1,2.7
2,6.9
3,4.5
4,1.5
5,3.2
6,4.8
7,3.5
8,2.5
9,8.5


In [15]:
y_train=y_train.iloc[:,0]

In [16]:
y_train

0     69.0
1     30.0
2     76.0
3     41.0
4     20.0
5     27.0
6     54.0
7     30.0
8     30.0
9     75.0
10    60.0
11    88.0
12    81.0
13    86.0
14    62.0
15    21.0
16    67.0
17    24.0
18    17.0
19    25.0
Name: Scores, dtype: float32

In [17]:
y_test=y_test.iloc[:,0]

In [18]:
sagemaker_session=sagemaker.Session()
bucket_name="vignesh-sagemaker-test"
prefix="linear-learner"
role=sagemaker.get_execution_role()

In [19]:
X_train = X_train.to_numpy()
y_train = y_train.to_numpy()

In [20]:
buf=io.BytesIO()
smac.write_numpy_to_dense_tensor(buf,X_train,y_train)
buf.seek(0)

0

In [21]:
key="student-data"
boto3.resource('s3').Bucket(bucket_name).Object(os.path.join(prefix,'train',key)).upload_fileobj(buf)
s3_train_data=f"s3://{bucket_name}/{prefix}/train/{key}"
print("Data Uploaded",s3_train_data)

Data Uploaded s3://vignesh-sagemaker-test/linear-learner/train/student-data


In [22]:
X_test=np.array(X_test)
buf=io.BytesIO()
smac.write_numpy_to_dense_tensor(buf,X_test,y_test)
buf.seek(0)
key="student-data-test"
boto3.resource('s3').Bucket(bucket_name).Object(os.path.join(prefix,'test',key)).upload_fileobj(buf)
s3_test_data=f"s3://{bucket_name}/{prefix}/test/{key}"
print("Data Uploaded",s3_test_data)

Data Uploaded s3://vignesh-sagemaker-test/linear-learner/test/student-data-test


In [25]:
output_location=f"s3://{bucket_name}/{prefix}/output"
output_location

's3://vignesh-sagemaker-test/linear-learner/output'

In [29]:
container = sagemaker.image_uris.retrieve(
    "linear-learner",
    boto3.Session().region_name
)
print(container)


382416733822.dkr.ecr.us-east-1.amazonaws.com/linear-learner:1


In [33]:
linear = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.c4.4xlarge",
    output_path=output_location,
    sagemaker_session=sagemaker_session
)

In [34]:
linear.set_hyperparameters(
    feature_dim=1,
    predictor_type="regressor",
    mini_batch_size=4,
    epochs=6,
    num_models=32,
    loss="absolute_loss"
)

In [35]:
linear.fit({"train": s3_train_data})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-03-15-11-34-09-211


2026-03-15 11:34:10 Starting - Starting the training job...
2026-03-15 11:34:25 Starting - Preparing the instances for training...
2026-03-15 11:35:04 Downloading - Downloading the training image......
2026-03-15 11:36:05 Training - Training image download completed. Training in progress....
2026-03-15 11:36:35 Uploading - Uploading generated training modelDocker entrypoint called with argument(s): train
Running default environment configuration script
[03/15/2026 11:36:28 INFO 139913908160320] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 

In [36]:
linear_regresor=linear.deploy(initial_instance_count=1,instance_type="ml.m4.xlarge")

INFO:sagemaker:Creating model with name: linear-learner-2026-03-15-11-38-32-040
INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-03-15-11-38-32-040
INFO:sagemaker:Creating endpoint with name linear-learner-2026-03-15-11-38-32-040


-------!

In [40]:
linear_regresor.serializer = sagemaker.serializers.CSVSerializer()
linear_regresor.deserializer = sagemaker.deserializers.JSONDeserializer()

In [41]:
results=linear_regresor.predict(X_test)

In [42]:
results

{'predictions': [{'score': 86.19189453125},
  {'score': 41.73332977294922},
  {'score': 53.06590270996094},
  {'score': 75.73104858398438},
  {'score': 37.37464904785156}]}

In [46]:
predictions = np.array([i["score"] for i in results["predictions"]])

In [47]:
predictions

array([86.19189453, 41.73332977, 53.06590271, 75.73104858, 37.37464905])